# Attribute Extraction — ResNet18 Multi-Head Training with Metrics

Ноутбук настроен под твою структуру Google Drive:

```text
MyDrive/
└── vkr/
    ├── data_clean/
    ├── images_for_training/
    ├── dataset_annotations.json
    └── processed_dataset.csv
```

Что есть внутри:
- ResNet18 pretrained ImageNet
- input 224x224
- AdamW, lr=3e-4, weight_decay=1e-4
- all-items разметка
- clean images из `data_clean`
- multi-head classification
- сохранение весов сразу на Google Drive
- tqdm progress bars
- сравнительные метрики по эпохам
- сохранение истории метрик в CSV


## 1. Imports

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
import torchvision.transforms as transforms

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score

from tqdm import tqdm


## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as e:
    print("Google Drive mount is available only in Google Colab.")
    print(e)


Mounted at /content/drive
Google Drive mounted.


## 3. Project Paths

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/vkr")

CLEAN_IMAGE_DIR = PROJECT_DIR / "data_clean"
DIRTY_IMAGE_DIR = PROJECT_DIR / "images_for_training"

ANNOTATIONS_PATH = PROJECT_DIR / "dataset_annotations.json"
CSV_PATH = PROJECT_DIR / "processed_dataset.csv"

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints_resnet18"
METRICS_DIR = PROJECT_DIR / "metrics_resnet18"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_CSV_PATH = METRICS_DIR / "training_metrics.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("CLEAN_IMAGE_DIR:", CLEAN_IMAGE_DIR)
print("DIRTY_IMAGE_DIR:", DIRTY_IMAGE_DIR)
print("ANNOTATIONS_PATH:", ANNOTATIONS_PATH)
print("CSV_PATH:", CSV_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("METRICS_CSV_PATH:", METRICS_CSV_PATH)


PROJECT_DIR: /content/drive/MyDrive/vkr
CLEAN_IMAGE_DIR: /content/drive/MyDrive/vkr/data_clean
DIRTY_IMAGE_DIR: /content/drive/MyDrive/vkr/images_for_training
ANNOTATIONS_PATH: /content/drive/MyDrive/vkr/dataset_annotations.json
CSV_PATH: /content/drive/MyDrive/vkr/processed_dataset.csv
CHECKPOINT_DIR: /content/drive/MyDrive/vkr/checkpoints_resnet18
METRICS_CSV_PATH: /content/drive/MyDrive/vkr/metrics_resnet18/training_metrics.csv


## 4. Check Files

In [ ]:
required_paths = {
    "PROJECT_DIR": PROJECT_DIR,
    "CLEAN_IMAGE_DIR": CLEAN_IMAGE_DIR,
    "ANNOTATIONS_PATH": ANNOTATIONS_PATH,
    "CSV_PATH": CSV_PATH,
}

for name, path in required_paths.items():
    print(f"{name}: {path} ->", "OK" if path.exists() else "NOT FOUND")


PROJECT_DIR: /content/drive/MyDrive/vkr -> OK
CLEAN_IMAGE_DIR: /content/drive/MyDrive/vkr/data_clean -> OK
ANNOTATIONS_PATH: /content/drive/MyDrive/vkr/dataset_annotations.json -> OK
CSV_PATH: /content/drive/MyDrive/vkr/processed_dataset.csv -> OK


## 5. Training Configuration

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 2

# Основной режим: clean images
IMAGE_DIR = CLEAN_IMAGE_DIR

# Threshold для multi-label design_features
DESIGN_THRESHOLD = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


## 6. Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


## 7. Dataset — all-items from dataset_annotations.json

In [ ]:
def normalize_list_value(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip() and str(v).strip().lower() != "null"]
    value = str(value).strip()
    if not value or value.lower() == "null":
        return []
    return [value]


class AttributeDataset(Dataset):
    def __init__(self, annotations_path, image_dir, transform=None):
        self.annotations_path = Path(annotations_path)
        self.image_dir = Path(image_dir)
        self.transform = transform

        with open(self.annotations_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows = []

        for image_path_from_json, items in data.items():
            image_name = Path(image_path_from_json).name

            full_image_path = self.image_dir / image_name

            if not full_image_path.exists():
                full_image_path = self.image_dir / image_path_from_json

            if not full_image_path.exists():
                continue

            for item_name, attrs in items.items():
                rows.append({
                    "image_path": str(full_image_path),
                    "source_image": image_path_from_json,
                    "item_name": item_name,
                    "item_type": attrs.get("item_type"),
                    "color": attrs.get("color"),
                    "season": attrs.get("season"),
                    "style": attrs.get("style"),
                    "design_features": normalize_list_value(attrs.get("design_features")),
                })

        self.df = pd.DataFrame(rows)

        if len(self.df) == 0:
            raise ValueError(
                "Dataset is empty. Check IMAGE_DIR and filenames in dataset_annotations.json."
            )

        self.label_encoders = {}

        for col in ["item_type", "color", "season", "style"]:
            self.df[col] = self.df[col].fillna("unknown").astype(str)
            encoder = LabelEncoder()
            self.df[col + "_id"] = encoder.fit_transform(self.df[col])
            self.label_encoders[col] = encoder

        self.design_mlb = MultiLabelBinarizer()
        design_matrix = self.design_mlb.fit_transform(self.df["design_features"])
        self.design_targets = torch.tensor(design_matrix, dtype=torch.float32)

        print("Total samples:", len(self.df))
        print("Images dir:", self.image_dir)
        print("item_type classes:", len(self.label_encoders["item_type"].classes_))
        print("color classes:", len(self.label_encoders["color"].classes_))
        print("season classes:", len(self.label_encoders["season"].classes_))
        print("style classes:", len(self.label_encoders["style"].classes_))
        print("design_features classes:", len(self.design_mlb.classes_))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        targets = {
            "item_type": torch.tensor(row["item_type_id"], dtype=torch.long),
            "color": torch.tensor(row["color_id"], dtype=torch.long),
            "season": torch.tensor(row["season_id"], dtype=torch.long),
            "style": torch.tensor(row["style_id"], dtype=torch.long),
            "design_features": self.design_targets[idx],
        }

        return image, targets


## 8. Create Dataset and DataLoaders

In [ ]:
full_dataset = AttributeDataset(
    annotations_path=ANNOTATIONS_PATH,
    image_dir=IMAGE_DIR,
    transform=train_transform,
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

val_base_dataset = AttributeDataset(
    annotations_path=ANNOTATIONS_PATH,
    image_dir=IMAGE_DIR,
    transform=val_transform,
)

val_dataset.dataset = val_base_dataset

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


Total samples: 62684
Images dir: /content/drive/MyDrive/vkr/data_clean
item_type classes: 76
color classes: 64
season classes: 8
style classes: 45
design_features classes: 532
Total samples: 62684
Images dir: /content/drive/MyDrive/vkr/data_clean
item_type classes: 76
color classes: 64
season classes: 8
style classes: 45
design_features classes: 532


## 9. Model — ResNet18 Multi-Head

In [ ]:
class MultiHeadResNet18(nn.Module):
    def __init__(self, num_item_type, num_color, num_season, num_style, num_design):
        super().__init__()

        backbone = torchvision.models.resnet18(weights="IMAGENET1K_V1")
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        in_features = backbone.fc.in_features

        self.item_type_head = nn.Linear(in_features, num_item_type)
        self.color_head = nn.Linear(in_features, num_color)
        self.season_head = nn.Linear(in_features, num_season)
        self.style_head = nn.Linear(in_features, num_style)
        self.design_head = nn.Linear(in_features, num_design)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)

        return {
            "item_type": self.item_type_head(x),
            "color": self.color_head(x),
            "season": self.season_head(x),
            "style": self.style_head(x),
            "design_features": self.design_head(x),
        }


## 10. Initialize Model

In [ ]:
num_item_type = len(full_dataset.label_encoders["item_type"].classes_)
num_color = len(full_dataset.label_encoders["color"].classes_)
num_season = len(full_dataset.label_encoders["season"].classes_)
num_style = len(full_dataset.label_encoders["style"].classes_)
num_design = len(full_dataset.design_mlb.classes_)

model = MultiHeadResNet18(
    num_item_type=num_item_type,
    num_color=num_color,
    num_season=num_season,
    num_style=num_style,
    num_design=num_design,
).to(DEVICE)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

## 11. Loss

In [ ]:
ce_loss = nn.CrossEntropyLoss()
bce_loss = nn.BCEWithLogitsLoss()

def compute_loss(outputs, targets):
    loss_item_type = ce_loss(outputs["item_type"], targets["item_type"])
    loss_color = ce_loss(outputs["color"], targets["color"])
    loss_season = ce_loss(outputs["season"], targets["season"])
    loss_style = ce_loss(outputs["style"], targets["style"])
    loss_design = bce_loss(outputs["design_features"], targets["design_features"].float())

    total_loss = (
        loss_item_type
        + loss_color
        + loss_season
        + loss_style
        + loss_design
    )

    return total_loss


## 12. Metrics

In [ ]:
def accuracy_from_logits(logits, targets):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == targets).sum().item()
    total = targets.numel()
    return correct, total


def calculate_validation_metrics(all_outputs, all_targets, threshold=DESIGN_THRESHOLD):
    metrics = {}

    # Single-label accuracy
    for head in ["item_type", "color", "season", "style"]:
        logits = torch.cat(all_outputs[head], dim=0)
        targets = torch.cat(all_targets[head], dim=0)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == targets).float().mean().item()

        metrics[f"{head}_accuracy"] = acc

    # Multi-label metrics for design_features
    design_logits = torch.cat(all_outputs["design_features"], dim=0)
    design_targets = torch.cat(all_targets["design_features"], dim=0)

    design_probs = torch.sigmoid(design_logits)
    design_preds = (design_probs >= threshold).int().cpu().numpy()
    design_true = design_targets.int().cpu().numpy()

    metrics["design_features_f1_micro"] = f1_score(
        design_true,
        design_preds,
        average="micro",
        zero_division=0,
    )
    metrics["design_features_f1_macro"] = f1_score(
        design_true,
        design_preds,
        average="macro",
        zero_division=0,
    )
    metrics["design_features_precision_micro"] = precision_score(
        design_true,
        design_preds,
        average="micro",
        zero_division=0,
    )
    metrics["design_features_recall_micro"] = recall_score(
        design_true,
        design_preds,
        average="micro",
        zero_division=0,
    )

    # Общая score-метрика для сравнения эпох.
    # Можно менять веса под приоритеты задачи.
    single_label_mean_acc = np.mean([
        metrics["item_type_accuracy"],
        metrics["color_accuracy"],
        metrics["season_accuracy"],
        metrics["style_accuracy"],
    ])

    metrics["single_label_mean_accuracy"] = float(single_label_mean_acc)
    metrics["overall_score"] = float(
        0.5 * single_label_mean_acc
        + 0.5 * metrics["design_features_f1_micro"]
    )

    return metrics


## 13. Save Checkpoints and Metrics to Google Drive

In [ ]:
def save_checkpoint_to_drive(
    model,
    optimizer,
    epoch,
    val_loss,
    metrics=None,
    checkpoint_dir=CHECKPOINT_DIR,
    best=False,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": float(val_loss),
        "metrics": metrics or {},
        "image_size": IMAGE_SIZE,
        "backbone": "resnet18",
        "pretrained_weights": "imagenet",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "label_encoders": {
            key: encoder.classes_.tolist()
            for key, encoder in full_dataset.label_encoders.items()
        },
        "design_features_classes": full_dataset.design_mlb.classes_.tolist(),
    }

    if best:
        checkpoint_path = checkpoint_dir / "best_model.pt"
    else:
        checkpoint_path = checkpoint_dir / f"model_epoch_{epoch:03d}.pt"

    torch.save(checkpoint, checkpoint_path)
    print(f"Saved checkpoint to Google Drive: {checkpoint_path}")

    return checkpoint_path


def save_final_weights_to_drive(
    model,
    checkpoint_dir=CHECKPOINT_DIR,
    filename="final_model_weights.pt",
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    weights_path = checkpoint_dir / filename
    torch.save(model.state_dict(), weights_path)

    print(f"Saved final model weights to Google Drive: {weights_path}")
    return weights_path


def save_metrics_history(metrics_history, csv_path=METRICS_CSV_PATH):
    df_metrics = pd.DataFrame(metrics_history)
    df_metrics.to_csv(csv_path, index=False)
    print(f"Saved metrics history to Google Drive: {csv_path}")
    return df_metrics


## 14. Train and Validate

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, targets in progress_bar:
        images = images.to(DEVICE)
        targets = {k: v.to(DEVICE) for k, v in targets.items()}

        optimizer.zero_grad()

        outputs = model(images)
        loss = compute_loss(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    return total_loss / max(len(loader), 1)


def validate_one_epoch(model, loader):
    model.eval()
    total_loss = 0.0

    all_outputs = {
        "item_type": [],
        "color": [],
        "season": [],
        "style": [],
        "design_features": [],
    }

    all_targets = {
        "item_type": [],
        "color": [],
        "season": [],
        "style": [],
        "design_features": [],
    }

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, targets in progress_bar:
            images = images.to(DEVICE)
            targets = {k: v.to(DEVICE) for k, v in targets.items()}

            outputs = model(images)
            loss = compute_loss(outputs, targets)

            total_loss += loss.item()

            for key in all_outputs.keys():
                all_outputs[key].append(outputs[key].detach().cpu())
                all_targets[key].append(targets[key].detach().cpu())

            progress_bar.set_postfix({
                "val_loss": f"{loss.item():.4f}"
            })

    avg_loss = total_loss / max(len(loader), 1)
    metrics = calculate_validation_metrics(all_outputs, all_targets)

    return avg_loss, metrics


## 15. Run Training with Comparative Metrics

In [ ]:
best_overall_score = -1.0
metrics_history = []

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_metrics = validate_one_epoch(model, val_loader)

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **val_metrics,
    }

    metrics_history.append(epoch_metrics)

    print(f"train_loss: {train_loss:.4f}")
    print(f"val_loss: {val_loss:.4f}")
    print(f"item_type_accuracy: {val_metrics['item_type_accuracy']:.4f}")
    print(f"color_accuracy: {val_metrics['color_accuracy']:.4f}")
    print(f"season_accuracy: {val_metrics['season_accuracy']:.4f}")
    print(f"style_accuracy: {val_metrics['style_accuracy']:.4f}")
    print(f"design_features_f1_micro: {val_metrics['design_features_f1_micro']:.4f}")
    print(f"design_features_f1_macro: {val_metrics['design_features_f1_macro']:.4f}")
    print(f"design_features_precision_micro: {val_metrics['design_features_precision_micro']:.4f}")
    print(f"design_features_recall_micro: {val_metrics['design_features_recall_micro']:.4f}")
    print(f"overall_score: {val_metrics['overall_score']:.4f}")

    save_checkpoint_to_drive(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        val_loss=val_loss,
        metrics=val_metrics,
        best=False,
    )

    # Лучший checkpoint теперь выбирается по overall_score, а не только по val_loss.
    if val_metrics["overall_score"] > best_overall_score:
        best_overall_score = val_metrics["overall_score"]

        save_checkpoint_to_drive(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            val_loss=val_loss,
            metrics=val_metrics,
            best=True,
        )

    # CSV обновляется после каждой эпохи
    metrics_df = save_metrics_history(metrics_history)

save_final_weights_to_drive(model)

metrics_df



Epoch 1/2


Training:   0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


train_loss: 6.6896
val_loss: 6.6349
item_type_accuracy: 0.2376
color_accuracy: 0.2370
season_accuracy: 0.4263
style_accuracy: 0.3948
design_features_f1_micro: 0.0000
design_features_f1_macro: 0.0000
design_features_precision_micro: 0.0000
design_features_recall_micro: 0.0000
overall_score: 0.1620
Saved checkpoint to Google Drive: /content/drive/MyDrive/vkr/checkpoints_resnet18/model_epoch_000.pt
Saved checkpoint to Google Drive: /content/drive/MyDrive/vkr/checkpoints_resnet18/best_model.pt
Saved metrics history to Google Drive: /content/drive/MyDrive/vkr/metrics_resnet18/training_metrics.csv

Epoch 2/2


Training:   0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


train_loss: 6.6187
val_loss: 6.6472
item_type_accuracy: 0.1803
color_accuracy: 0.2370
season_accuracy: 0.4263
style_accuracy: 0.3948
design_features_f1_micro: 0.0000
design_features_f1_macro: 0.0000
design_features_precision_micro: 0.0000
design_features_recall_micro: 0.0000
overall_score: 0.1548
Saved checkpoint to Google Drive: /content/drive/MyDrive/vkr/checkpoints_resnet18/model_epoch_001.pt
Saved metrics history to Google Drive: /content/drive/MyDrive/vkr/metrics_resnet18/training_metrics.csv
Saved final model weights to Google Drive: /content/drive/MyDrive/vkr/checkpoints_resnet18/final_model_weights.pt


,epoch,train_loss,val_loss,item_type_accuracy,color_accuracy,season_accuracy,style_accuracy,design_features_f1_micro,design_features_f1_macro,design_features_precision_micro,design_features_recall_micro,single_label_mean_accuracy,overall_score
0,0,6.689565,6.634930,0.237617,0.236979,0.426258,0.394831,0.0,0.0,0.0,0.0,0.323921,0.161961
1,1,6.618650,6.647193,0.180266,0.236979,0.426258,0.394831,0.0,0.0,0.0,0.0,0.309584,0.154792


## 16. View Metrics Table

In [ ]:
metrics_df = pd.read_csv(METRICS_CSV_PATH)
metrics_df


,epoch,train_loss,val_loss,item_type_accuracy,color_accuracy,season_accuracy,style_accuracy,design_features_f1_micro,design_features_f1_macro,design_features_precision_micro,design_features_recall_micro,single_label_mean_accuracy,overall_score
0,0,6.689565,6.634930,0.237617,0.236979,0.426258,0.394831,0.0,0.0,0.0,0.0,0.323921,0.161961
1,1,6.618650,6.647193,0.180266,0.236979,0.426258,0.394831,0.0,0.0,0.0,0.0,0.309584,0.154792


## 17. Load Saved Weights from Google Drive

In [ ]:
# Полный checkpoint:
# checkpoint_path = CHECKPOINT_DIR / "best_model.pt"
# checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Только финальные веса:
# weights_path = CHECKPOINT_DIR / "final_model_weights.pt"
# model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
